In [1]:
%pip install -r requirements-offline.txt --quiet

/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-29/.check-env/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Solution — Module M4: เสียง → ตัวเลข → RF (ESC-50)

เฉลยของ `exercise.ipynb` — ตัวเลขอ้างอิงจากรอบที่รันด้วย seed 42
(ต่างกันได้เล็กน้อยตามเครื่อง/เวอร์ชัน library)

## 0. เตรียมข้อมูล (ให้มาแล้ว — รันผ่าน)

In [2]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

from pathlib import Path  # noqa: E402


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root "
        "(uv run jupyter lab)"
    )


REPO = repo_root()


In [3]:
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import requests  # noqa: E402
import zipfile  # noqa: E402
import librosa  # noqa: E402

ESC50_URL = "https://github.com/karoldvl/ESC-50/archive/master.zip"


def find_esc50() -> Path:
    """ค้นหาโฟลเดอร์ ESC-50 ที่แตกไว้แล้วจากตำแหน่งที่รู้จัก."""
    env_dir = os.environ.get("ESC50_DIR")
    candidates = [
        Path(env_dir) if env_dir else None,
        REPO / "datasets" / "esc50" / "ESC-50-master",
        Path.home() / ".cache" / "esc50" / "ESC-50-master",
        Path.home() / "Downloads" / "ESC-50-master",
    ]
    for c in candidates:
        if c is not None and (c / "meta" / "esc50.csv").exists():
            print("พบ ESC-50 ที่:", c)
            return c
    return None


ESC50 = find_esc50()
if ESC50 is None:
    cache = Path.home() / ".cache" / "esc50"
    cache.mkdir(parents=True, exist_ok=True)
    zpath = cache / "esc50-master.zip"
    if not zpath.exists():
        print(f"ดาวน์โหลด ESC-50 (~600 MB) จาก {ESC50_URL}")
        with requests.get(ESC50_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zpath, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(cache)
    ESC50 = cache / "ESC-50-master"


พบ ESC-50 ที่: /Users/utarn/.cache/esc50/ESC-50-master


In [4]:
meta = pd.read_csv(ESC50 / "meta" / "esc50.csv")
id2cat = (
    meta.drop_duplicates("target")
    .sort_values("target")["category"]
    .tolist()
)
paths = [ESC50 / "audio" / f for f in meta["filename"]]
y_all = meta["target"].to_numpy()
folds = meta["fold"].to_numpy()
print("จำนวนคลิป:", len(meta), "| จำนวน class:",
      meta["category"].nunique())
print(meta["fold"].value_counts().sort_index().to_string())
print(meta["category"].value_counts().head(8).to_string())


จำนวนคลิป: 2000 | จำนวน class: 50
fold
1    400
2    400
3    400
4    400
5    400
category
dog                40
chirping_birds     40
vacuum_cleaner     40
thunderstorm       40
door_wood_knock    40
can_opening        40
crow               40
clapping           40


In [5]:
SR = 22050


def mfcc_row(path: Path) -> np.ndarray:
    """คลิปเดียว → เวกเตอร์ 40 ตัวเลข (mean+std ของ MFCC 20 ตัว)."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20)
    return np.concatenate([m.mean(axis=1), m.std(axis=1)])


def evaluate_folds(Xa, ya, fa, n_est=200):
    """วน 5-fold ทางการของ ESC-50 — คืน dict {fold: accuracy}."""
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score

    out = {}
    for k in range(1, 6):
        tr, te = fa != k, fa == k
        clf = RandomForestClassifier(
            n_estimators=n_est, n_jobs=-1, random_state=SEED
        )
        clf.fit(Xa[tr], ya[tr])
        out[k] = accuracy_score(ya[te], clf.predict(Xa[te]))
        print(f"  fold {k}: accuracy = {out[k]:.4f}")
    print(f"  mean = {float(np.mean(list(out.values()))):.4f}")
    return out


from joblib import Parallel, delayed  # noqa: E402

X = np.asarray(Parallel(n_jobs=-1)(
    delayed(mfcc_row)(p) for p in paths
))
print("X:", X.shape)
accs = evaluate_folds(X, y_all, folds)

X: (2000, 40)


  fold 1: accuracy = 0.4350


  fold 2: accuracy = 0.4375


  fold 3: accuracy = 0.4600


  fold 4: accuracy = 0.4725


  fold 5: accuracy = 0.4675
  mean = 0.4545


## ข้อ 1 — delta-MFCC: feature 80 ตัว

In [6]:
def mfcc_delta_row(path: Path) -> np.ndarray:
    """คลิปเดียว → เวกเตอร์ 80 ตัวเลข (MFCC + delta, mean+std)."""
    y, _ = librosa.load(path, sr=SR, mono=True)
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20)
    d = librosa.feature.delta(m, axis=1)
    return np.concatenate([
        m.mean(axis=1), m.std(axis=1),
        d.mean(axis=1), d.std(axis=1),
    ])


X2 = np.asarray(Parallel(n_jobs=-1)(
    delayed(mfcc_delta_row)(p) for p in paths
))
print("X2:", X2.shape)
accs2 = evaluate_folds(X2, y_all, folds)

X2: (2000, 80)


  fold 1: accuracy = 0.4625


  fold 2: accuracy = 0.4825


  fold 3: accuracy = 0.4775


  fold 4: accuracy = 0.5200


  fold 5: accuracy = 0.4850
  mean = 0.4855


**คำตอบข้อ 1:** จากรอบที่รัน (seed 42): baseline 40 ตัวเลข = **0.4545**,
เพิ่ม delta เป็น 80 ตัวเลข = **0.4855** — ดีขึ้น ~3 จุด คุ้มค่า เพราะ delta จับ
"จังหวะเสียงเคลื่อนไหว" (เช่น เสียงสัตว์ร้องต่อเนื่อง vs เสียงกระแทกครั้งเดียว)
ที่ mean/std เดิมมองไม่เห็น แต่สังเกตด้วยว่ายังห่างจาก CNN ที่อ่าน spectrogram
เต็มอยู่มาก — feature เพิ่มได้ แต่เพดานของ "สรุปเป็นตัวเลขนิ่ง ๆ" ยังต่ำ

## ข้อ 2 — n_estimators: 50 / 200 / 500

In [7]:
acc_50 = evaluate_folds(X, y_all, folds, n_est=50)
acc_500 = evaluate_folds(X, y_all, folds, n_est=500)

  fold 1: accuracy = 0.4150
  fold 2: accuracy = 0.3875


  fold 3: accuracy = 0.4075


  fold 4: accuracy = 0.4375
  fold 5: accuracy = 0.4225
  mean = 0.4140


  fold 1: accuracy = 0.4375


  fold 2: accuracy = 0.4425


  fold 3: accuracy = 0.4575


  fold 4: accuracy = 0.4825


  fold 5: accuracy = 0.4800
  mean = 0.4600


**คำตอบข้อ 2:** จากรอบที่รัน: 50 ต้น = **0.414**, 200 ต้น = **0.4545**,
500 ต้น = **0.4600** — เพิ่มจาก 50 → 200 ได้ ~4 จุด แต่ 200 → 500 ได้แค่ ~0.5 จุด
(แลกกับเวลาเทรน ~2.5 เท่า) — 200 อยู่บน "จุดหักศอก" ของช่วงค่าตอบแทนต่อเวลา
แบบเดียวกับ hyperparameter อื่น ๆ ที่คุณเคยวนหาในคอร์ส DS: เพิ่มเติมเมื่อโมเดล
ใหญ่ขึ้นไม่ได้แปลว่าดีขึ้นเสมอ

## ข้อ 3 — คู่สับสนของ fold 5

In [8]:
from collections import Counter  # noqa: E402
from sklearn.ensemble import (  # noqa: E402
    RandomForestClassifier,
)

te5 = folds == 5
rf5 = RandomForestClassifier(
    n_estimators=200, n_jobs=-1, random_state=SEED
).fit(X[folds != 5], y_all[folds != 5])
pred5 = rf5.predict(X[te5])

wrong = Counter(
    (id2cat[t], id2cat[p])
    for t, p in zip(y_all[te5], pred5) if t != p
)
print(wrong.most_common(8))

[(('breathing', 'keyboard_typing'), 4), (('siren', 'church_bells'), 3), (('frog', 'dog'), 3), (('helicopter', 'wind'), 3), (('insects', 'sea_waves'), 3), (('drinking_sipping', 'keyboard_typing'), 3), (('coughing', 'sneezing'), 3), (('laughing', 'crying_baby'), 3)]


**คำตอบข้อ 3:** จากรอบที่รัน คู่บ่อยสุด: `breathing → keyboard_typing`
(4 ครั้ง), `siren → church_bells`, `frog → dog`, `helicopter → wind`,
`insects → sea_waves`, `coughing → sneezing`, `laughing → crying_baby` —
สมเหตุสมผลทางเสียง: เสียงหายใจและเสียงคลิกคีย์บอร์ดเป็น "จังหวะสั้น ๆ ซ้ำ ๆ"
ส่วนไซเรน/ระฆังเป็น "โทนยาวต่อเนื่อง" — หลังบีบเป็น mean/std แล้วตัวเลข MFCC
ของสองพวกนี้ใกล้กันมาก RF (และหูมนุษย์ที่ต้องสะสมหลักฐาน) เลยแยกได้ยาก

## ข้อ 4 — ESC-10: โจทย์แคบลง

In [9]:
mask = meta["esc10"].to_numpy()
accs10 = evaluate_folds(X[mask], y_all[mask], folds[mask])

  fold 1: accuracy = 0.7125


  fold 2: accuracy = 0.7625


  fold 3: accuracy = 0.8375


  fold 4: accuracy = 0.8000


  fold 5: accuracy = 0.7375
  mean = 0.7700


**คำตอบข้อ 4:** จากรอบที่รัน: ESC-10 accuracy เฉลี่ย = **0.77**
(0.7125–0.8375 ต่อ fold) — สูงกว่าโจทย์ 50 class (0.4545) กว่า 1.7 เท่า
ทั้งที่ใช้ feature และโมเดลเดิมทุกอย่าง ข้อสังเกตสำคัญ: การเดาสุ่มเปลี่ยนจาก
1/50 = 2% เป็น 1/10 = 10% แต่ 0.77 สูงกว่าการเดาสุ่มแบบเดียวกันกับที่ 0.4545
สูงกว่า 2% — คลาสที่เลือกมาใน ESC-10 (สัตว์ 10 ตัว, เสียงที่แตกต่างชัด) แยกได้
ง่ายกว่าจริง ๆ บทเรียนเมโทรโลยี: **แคบโจทย์ให้ตรงความจำเป็น แล้วโมเดลเดิม
ทำงานได้ดีขึ้นทันที**

## ข้อ 5 — เครื่องจักร 200 คลิป ไม่มี GPU

**คำตอบข้อ 5:** เลือก **RF บน MFCC** ก่อน เหตุผลเฉพาะโจทย์:
1. **200 คลิปน้อยเกินสำหรับ CNN** — จากข้อ 4 เราเห็นว่าโจทย์แคบ (2 class)
   ทำให้ feature ธรรมดาแยกได้ดีแล้ว ส่วน CNN ที่ต้องเรียนรู้ pattern เองจาก
   200 ตัวอย่างมัก overfit ก่อนจะมีประโยชน์ (M1 เจอเรื่องเดียวกันใน learning
   curve — ข้อมูล 5% แล้ว transfer learning ชนะ, CNN จากศูนย์จม)
2. **เครื่องปลายทางไม่มี GPU** — RF วิ่งบน CPU ได้ทันที อนุมานต่อคลิปในหลัก
   มิลลิวินาที ขณะที่ CNN ต้องหาทาง deploy บน GPU/quantize ซึ่งเป็นต้นทุนใหญ่
   ของโครงการจริง
3. **ตรวจสอบได้** — งานเมโทรโลยีต้องตอบได้ว่าทำไมโมเดลตัดสินใจแบบนั้น
   feature importance ของ RF บอกได้ (เช่น "โมเดลดู energy ช่วงความถี่ต่ำ")
   ขณะที่ CNN เป็นกล่องดำ (แก้ได้ด้วย Grad-CAM — M7 จะสอน)

ถ้า RF ยังไม่พอในข้อจำกัดที่วัดได้ ค่อยพิจารณา CNN — และระวัง license ของ
ข้อมูลเสริมที่จะเอามาใช้ (ถ้าแจกต่อเชิงพาณิชย์ ให้เลี่ยงชุดแบบ ESC-50)

## สรุป (เฉลย)

- delta-MFCC: 0.4545 → **0.4855** (+3 จุด) — จับความเปลี่ยนแปลงตามเวลาได้
- ต้นไม้ 200 = จุดหักศอก (50 → 0.414, 500 → 0.4600)
- คู่สับสน = เสียงที่ "สรุปด้วย mean/std แล้วเหมือนกัน" ไม่ใช่เสียงที่หูฟัง
  แยกไม่ออกเสมอไป
- ESC-10 = **0.77** — แคบโจทย์ = โมเดลเดิมเก่งขึ้น
- โจทย์จริงที่ข้อมูลน้อย + ไม่มี GPU + ต้องตรวจสอบได้ → RF ก่อนเสมอ